# CGC_STAGE=1 PPO (env-v3)

실행 중 보조 규칙 없음. 보상은 `train.py`의 B-다음 칸(속도 0, green_stop 0.05/0.10, 놓침 −20).

Internet 켜기. Accelerator는 CPU. 이 노트북만 연다.
비공개 저장소면 Secrets에 `GITHUB_TOKEN`.

**12시간 컷 → resume 체인**
1. 세션이 끝나면 Output에서 `exp-ppo-stage1` 폴더 또는 tar를 받는다.
2. 다음 세션에서 Add data로 그 출력을 붙이거나 `/kaggle/working/exp-ppo-stage1/model.resume.pt`가 남았는지 본다.
3. 학습 칸은 `model.resume.pt`가 있으면 `--resume`을 붙인다. `--total-steps`는 최종 누적(기본 300000)이지 이번 세션 추가량이 아니다.

푸시 전에는 Kaggle clone이 이 브랜치를 못 받는다. 푸시 후 아래 `COMMIT`를 그 해시로 고정한다.

In [ ]:
import os, subprocess
ROOT = "/kaggle/working"
SUMO_SRC = f"{ROOT}/sumo-1.27.1"
SUMO_PREFIX = f"{ROOT}/sumo-install"
os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(f"{SUMO_PREFIX}/bin/sumo"):
    subprocess.check_call("apt-get update && apt-get install -y cmake g++ libxerces-c-dev libgdal-dev libproj-dev curl", shell=True)
    archive = f"{ROOT}/sumo-src-1.27.1.tar.gz"
    subprocess.check_call(["curl", "-L", "-o", archive, "https://sumo.dlr.de/releases/1.27.1/sumo-src-1.27.1.tar.gz"])
    subprocess.check_call(["tar", "-xzf", archive, "-C", ROOT])
    build = f"{SUMO_SRC}/build"
    os.makedirs(build, exist_ok=True)
    subprocess.check_call(["cmake", "-S", SUMO_SRC, "-B", build, "-DCMAKE_BUILD_TYPE=Release", f"-DCMAKE_INSTALL_PREFIX={SUMO_PREFIX}"])
    subprocess.check_call(["cmake", "--build", build, "-j2"])
    subprocess.check_call(["cmake", "--install", build])
os.environ["SUMO_HOME"] = SUMO_PREFIX
os.environ["PATH"] = f"{SUMO_PREFIX}/bin:" + os.environ.get("PATH", "")
print(subprocess.check_output(["sumo", "--version"], text=True).splitlines()[0])

In [ ]:
import os, subprocess, shutil
COMMIT = ""  # 푸시 후 커밋 해시를 넣는다. 비우면 브랜치 tip.
BRANCH = "model/exp-ppo-stage1"
REPO = "/kaggle/working/repo-stage1"
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("GITHUB_TOKEN 없음:", exc)
url = "https://github.com/TranserZ/Artificial-Intelligence-and-Control-for-Autonomous-Driving.git"
if token:
    url = url.replace("https://", f"https://{token}@")
if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "--branch", BRANCH, url, REPO])
else:
    subprocess.check_call(["git", "fetch", "origin", BRANCH], cwd=REPO)
    subprocess.check_call(["git", "checkout", BRANCH], cwd=REPO)
    subprocess.check_call(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO)
if COMMIT:
    subprocess.check_call(["git", "checkout", COMMIT], cwd=REPO)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True))

OUT = "/kaggle/working/exp-ppo-stage1"
os.makedirs(OUT, exist_ok=True)
resume_name = "model.resume.pt"
if not os.path.exists(os.path.join(OUT, resume_name)):
    for root, _dirs, files in os.walk("/kaggle/input"):
        if resume_name in files:
            src = os.path.join(root, resume_name)
            shutil.copy2(src, os.path.join(OUT, resume_name))
            print("입력에서 resume 복사", src)
            break

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "gymnasium==1.4.0", "matplotlib==3.11.2", "numpy==2.5.4",
    "tensorboard==2.21.0", "traci==1.28.0"])
import torch, traci
print("torch", torch.__version__, "traci", traci.__version__)

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/repo-stage1"
OUT = "/kaggle/working/exp-ppo-stage1"
os.makedirs(OUT, exist_ok=True)
env = os.environ.copy()
env["CGC_STAGE"] = "1"
cmd = [sys.executable, "train.py",
       "--seed", "0", "--replay-prob", "0", "--eval-interval", "0",
       "--total-steps", "300000", "--save-steps", "300000",
       "--out-dir", OUT]
resume = os.path.join(OUT, "model.resume.pt")
if os.path.exists(resume):
    cmd += ["--resume", resume]
    print("이어서 학습", resume)
subprocess.check_call(cmd, cwd=REPO, env=env)

In [ ]:
import os, subprocess
OUT = "/kaggle/working/exp-ppo-stage1"
archive = "/kaggle/working/exp-ppo-stage1.tar.gz"
subprocess.check_call(["tar", "-C", "/kaggle/working", "-czf", archive, "exp-ppo-stage1"])
print(archive, os.path.getsize(archive))
for name in ("config.json", "training_log.csv", "final.pt", "final.resume.pt",
             "step300k.pt", "model.resume.pt", "model.pt"):
    path = os.path.join(OUT, name)
    print(name, "있음" if os.path.exists(path) else "없음")